# Create Lasker Awards (PRIZE PATTERN)

Lasker Awards from laskerfoundation.org via the WordPress REST API
(`winners` custom post type with `?_embed=1`). One award post lists
the achievement with N laureates as a `winners_name` taxonomy.

**Awarding body:** Lasker Foundation — F4320311370

**Prerequisites:**
- Run `scripts/local/lasker_to_s3.py` first.

**S3:** `s3a://openalex-ingest/awards/lasker/lasker_awards.parquet`

Schema notes:
- One row per (award × laureate). Year-wrapper posts are skipped at
  ingest time.
- Lasker Awards have a $250K honorarium per category (BASIC,
  CLINICAL, SPECIAL ACHIEVEMENT) but the per-laureate share is not
  published. amount = NULL, currency = "USD". Step 6.7 waived.
- description = the achievement title (e.g. "AlphaFold—for predicting
  protein structures").
- funder_scheme = award category (BASIC / CLINICAL / SPECIAL
  ACHIEVEMENT).


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lasker_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lasker/lasker_awards.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.lasker_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.lasker_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.lasker_raw LIMIT 5;

## Step 2: Transform

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lasker_awards
USING delta
AS
WITH lasker_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320311370
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':lasker:', CAST(s.wp_post_id AS STRING),
                        ':', CAST(s.laureate_term_id AS STRING)))) % 9000000000 AS id,
    CONCAT('Lasker ', s.award_name, ' Award ', CAST(s.year AS STRING),
           ' — ', s.laureate_name) AS display_name,
    NULLIF(s.achievement_title, '') AS description,
    f.funder_id,
    CONCAT(CAST(s.wp_post_id AS STRING), '-', CAST(s.laureate_term_id AS STRING)) AS funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    s.award_name AS funder_scheme,           -- BASIC / CLINICAL / SPECIAL ACHIEVEMENT
    'lasker_wp' AS provenance,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(s.year AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
    s.year AS start_year,
    s.year AS end_year,
    struct(
        s.laureate_given_name AS given_name,
        s.laureate_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            -- Affiliation parsed by the script from the award post's <p.aw-name>+<p.aw-work> pairs.
            -- ~95% coverage; some "formerly at X" entries flow through verbatim.
            NULLIF(s.affiliation, '') AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    s.url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':lasker:', CAST(s.wp_post_id AS STRING),
                                ':', CAST(s.laureate_term_id AS STRING)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.lasker_raw s
CROSS JOIN lasker_funder f
WHERE s.laureate_name IS NOT NULL AND s.year IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 48

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lasker_wp' AND priority = 48;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id,
       amount, currency, funder, funding_type, funder_scheme, provenance,
       start_date, end_date, start_year, end_year,
       lead_investigator, co_lead_investigator, investigators,
       landing_page_url, doi, works_api_url,
       created_date, updated_date,
       48 AS priority
FROM openalex.awards.lasker_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(amount) has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) pct_amount,
       collect_set(currency) currencies,
       collect_set(funder_scheme) categories
FROM openalex.awards.lasker_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title, funder_scheme AS award,
       start_year, lead_investigator.given_name, lead_investigator.family_name
FROM openalex.awards.lasker_awards ORDER BY start_year DESC LIMIT 12;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) cnt FROM openalex.awards.lasker_awards GROUP BY funder_scheme;